In [11]:
import numpy as np
import matplotlib.pyplot as plt

import pyRTC.utils as utils
from pyRTC import *
from astropy.io import fits

In [2]:
shm_names = ["wfs", "wfsRaw", "wfc", "wfc2D", "wfcShape", "signal", "signal2D", "psfShort", "psfLong", "wfsInfo", "loop", "refSlopes", "subApMasks", "cmat", "m2c"] #list of SHMs to reset
clear_shms(shm_names)

Opening Existing Shared Memory Object wfs
Opening Existing Shared Memory Object wfs_meta
Opening Existing Shared Memory Object wfs_meta
Closing wfs
Creating New Shared Memory Object wfs_gpu_handle
Opening Existing Shared Memory Object wfs_gpu_handle_meta
Closing wfs_meta
Opening Existing Shared Memory Object wfsRaw
Opening Existing Shared Memory Object wfsRaw_meta
Closing wfs_gpu_handle
Opening Existing Shared Memory Object wfsRaw_meta
Closing wfsRaw
Creating New Shared Memory Object wfsRaw_gpu_handle
Opening Existing Shared Memory Object wfsRaw_gpu_handle_meta
Closing wfsRaw_meta
Creating New Shared Memory Object wfc
Creating New Shared Memory Object wfc_meta
Closing wfsRaw_gpu_handle
Opening Existing Shared Memory Object wfc_meta
Closing wfc
Creating New Shared Memory Object wfc_gpu_handle
Opening Existing Shared Memory Object wfc_gpu_handle_meta
Closing wfc_meta
Creating New Shared Memory Object wfc2D
Creating New Shared Memory Object wfc2D_meta
Closing wfc_gpu_handle
Opening Existi

In [18]:
from pathlib import Path
import os
# PYRTC_CLASS_PATH = "/Users/ellenlee/Documents/pyRTC-IRTF/pyRTC"
# CONFIG_PATH = "/Users/ellenlee/Documents/pyRTC-IRTF/IRTF/config"

PYRTC_CLASS_PATH = "/home/felix/src/pyrtc/pyRTC-IRTF/pyRTC"
CONFIG_PATH = "/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/config"

with open(os.path.join(CONFIG_PATH, "ports.json")) as f:
    PORTS = json.load(f)


def check_hardware_and_launch(hardware_class, config, port_name):
    # Need to explicilty check these files or the GUI will hang itself
    if not os.path.exists(hardware_class):
        raise FileNotFoundError(f"Hardware class file {hardware_class} not found.")
    if not os.path.exists(config):
        raise FileNotFoundError(f"Config file {config} not found.")
    return hardwareLauncher(hardware_class, config, port=PORTS[port_name])


def get_felixsim():
    hardware_class = os.path.join(PYRTC_CLASS_PATH, "hardware", "FELIXsim.py")
    config = os.path.join(CONFIG_PATH, "hrtc_wfs_felixsim.yaml")
    launcher = check_hardware_and_launch(hardware_class, config, "wfs")
    return launcher

def get_andor():
    hardware_class = os.path.join(PYRTC_CLASS_PATH, "hardware", "AndorWFS.py")
    config = os.path.join(CONFIG_PATH, "hrtc_wfs_andor.yaml")
    launcher = check_hardware_and_launch(hardware_class, config, "wfs")
    return launcher

In [19]:
wfs = get_andor()

In [20]:
wfs.launch()

Launching Process: /home/felix/src/pyrtc/pyRTC-IRTF/pyRTC/hardware/AndorWFS.py
Waiting for Process at 127.0.0.1:10489
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...
Connection failed: [Errno 111] Connection refused
Retrying in 2 seconds...


Connected


In [25]:
wfs.run("stop")

1

In [26]:
wfs.run("showAvailableReadout")

-1

In [23]:
wfs.run("setExposure", 0.005)

1

In [9]:
wfs.shutdown()

1

In [12]:
conf = utils.read_yaml_file("/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/config/hrtc_wfs_felixsim.yaml")
conf = utils.read_yaml_file("/home/felix/src/pyrtc/pyRTC-IRTF/IRTF/config/hrtc_wfs_andor.yaml")

wfs_conf = conf["wfs"]
# slopes_conf = conf["slopes"]
# dm_conf = conf["wfc"]
# loop_conf = conf["loop"]

#wfs = FELIXSimulator(wfs_conf)
wfs = AndorWFS(wfs_conf)
# slopes = SlopesProcess(conf=slopes_conf)
# dm = ImakaDM(dm_conf)
# loop = Loop(loop_conf)

Opening Existing Shared Memory Object wfsRaw
Opening Existing Shared Memory Object wfsRaw_meta
Thread expose: Priority set to REALTIME
Opening Existing Shared Memory Object wfs
Opening Existing Shared Memory Object wfs_meta
Opening Existing Shared Memory Object wfsInfo
Opening Existing Shared Memory Object wfsInfo_meta
Using HSSpeed index 0: 17.00 MHz
Using VSSpeed index 1: 0.50 µs


In [5]:
wfs.start()

In [13]:
wfs.stop()

In [7]:
wfs.sdk.AbortAcquisition()

20002

In [10]:
wfs.sdk.StartAcquisition()

20002

In [14]:
wfs.showAvailableReadout()

Function GetNumberADChannels returned 20002 number of available channels 1
Function GetNumberHSSpeeds 20002 number of available speeds 4
Available HSSpeeds in MHz [17.0, 10.0, 5.0, 1.0] 
Function GetNumberVSSpeeds 20002 number of available speeds 5
Available VSSpeeds in us [0.3, 0.5, 0.9, 1.7, 3.3]
Recommended VSSpeed 3.3 index 4
Function GetNumberAmp returned 20002 number of amplifiers 2
Available amplifier modes ['ElectronMultiplying', 'Conventional']


{'ADchannels': 1,
 'HSSpeeds': [17.0, 10.0, 5.0, 1.0],
 'VSSpeeds': [0.3, 0.5, 0.9, 1.7, 3.3],
 'VSSpeedRecommended': {'index': 4, 'speed': 3.3},
 'AmpModes': ['ElectronMultiplying', 'Conventional']}

In [7]:
wfs.sdk.GetBitDepth(0)

(20002, 16)

In [11]:
wfs.setReadout(hi=0, vi=1, ADChannel=0, amplifier=1)

Using HSSpeed index 0: 17.00 MHz
Using VSSpeed index 1: 0.50 µs


In [14]:
wfs.setExposure(0.01)

In [34]:
print(wfs.vi)

1


In [16]:
del(wfs)

In [17]:
wfs.stop()

NameError: name 'wfs' is not defined